In [24]:
import cv2
import os
import torch
import clip
from PIL import Image

In [25]:
def crop_image(crop_percent,img_path,output_folder):
    img_name = img_path.split("/")[-1]
    
    if not (img_name.endswith(".jpg") or img_name.endswith(".png")):
        print("image not cropped \n only .png or .jpg supported")
        return 
    
    img = cv2.imread(img_path)
    h,w = img.shape[:2]
    
    # compute crop coordinates
    top = int(h * crop_percent)
    bottom = int(h * (1 - crop_percent))
    left = int(w * crop_percent)
    right = int(w * (1 - crop_percent))
    
    
    
    cropped_img = img[top:bottom, left:right]
    print(f"{img_name}: cropped size = {cropped_img.shape}")

    name, ext = os.path.splitext(img_name)
    save_path = os.path.join(output_folder, f"{name}_{crop_percent}_pct_crop{ext}")
    success = cv2.imwrite(save_path, cropped_img)
    if not success:
        print(f"Failed to save {save_path}")

    print(f"{img_name} successfully cropped \n")
    return cropped_img
    

In [ ]:
def blur_image(kernel_size,sigmaX,img_path,output_folder):
    img_name = img_path.split("/")[-1]
    
    if not (img_name.endswith(".jpg") or img_name.endswith(".png")):
        print("image not cropped \n only .png or .jpg supported")
        return 
    
    img = cv2.imread(img_path)
    blurred_img = cv2.GaussianBlur(img,kernel_size,sigmaX)
    
    os.makedirs(output_folder, exist_ok=True)
    name, ext = os.path.splitext(img_name)
    save_path = os.path.join(output_folder, f"{name}_blurred_{ext}")
    success = cv2.imwrite(save_path, blurred_img)
    if not success:
        print(f"Failed to save {save_path}")

    print(f"{img_name} successfully blurred \n")
    
    return blurred_img
    
    

In [27]:
input_folder = "images/original/"
output_folder = "images/cropped/"
os.makedirs(output_folder, exist_ok=True)

In [28]:
crop_percent = 0.1
for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    crop_image(crop_percent, img_path, output_folder)

dog_02.png: cropped size = (342, 512, 3)
dog_02.png successfully cropped 

dog_01.png: cropped size = (342, 512, 3)
dog_01.png successfully cropped 

dog_03.png: cropped size = (384, 512, 3)
dog_03.png successfully cropped 

dog_05.png: cropped size = (258, 512, 3)
dog_05.png successfully cropped 

image not cropped 
 only .png or .jpg supported
image not cropped 
 only .png or .jpg supported
dog_04.png: cropped size = (512, 384, 3)
dog_04.png successfully cropped 



In [29]:
kernel_size = (5,5)
sigmaX = 0
blurred_output_folder= "images/blurred"
for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    blur_image(kernel_size,sigmaX,img_path,blurred_output_folder)

dog_02.png successfully cropped 

dog_01.png successfully cropped 

dog_03.png successfully cropped 

dog_05.png successfully cropped 

image not cropped 
 only .png or .jpg supported
image not cropped 
 only .png or .jpg supported
dog_04.png successfully cropped 



### similarity measure
*   Loops over images + captions
*   Computes similarity for original vs cropped
*   Computes drop in similarity (misalignment metric)

In [30]:
import sys
sys.path.append("images/original")  
from caption import captions  

original_folder = "images/original/"
cropped_folder = "images/cropped/"
blurred_folder= "images/blurred"


# Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def compute_similarity(image_path, text):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).item()
    return similarity

# Collect results
results = {}

for img_name, caption in captions.items():
    # Original image
    orig_path = os.path.join(original_folder, img_name)
    orig_sim = compute_similarity(orig_path, caption)
    
    # Cropped image
    name, ext = os.path.splitext(img_name)
    crop_name = f"{name}_{crop_percent}_pct_crop{ext}"
    crop_path = os.path.join(cropped_folder, crop_name)
    crop_sim = compute_similarity(crop_path, caption)
    
    results[img_name] = {
        "original": orig_sim,
        "cropped": crop_sim,
        "drop": orig_sim - crop_sim
    }

# Print results
for img_name, res in results.items():
    print(f"{img_name}: Original={res['original']:.4f}, Cropped={res['cropped']:.4f}, Drop={res['drop']:.4f}")


dog_04.png: Original=0.2953, Cropped=0.2882, Drop=0.0071
dog_03.png: Original=0.2877, Cropped=0.3023, Drop=-0.0146
dog_02.png: Original=0.2476, Cropped=0.2456, Drop=0.0020
dog_05.png: Original=0.3360, Cropped=0.3235, Drop=0.0125
dog_01.png: Original=0.2665, Cropped=0.2633, Drop=0.0032
